# Notebook 6 — Dispatcher fingerprinting on a live vLLM testbed
**Question.** Notebook 5 showed that pools of keys driven by one client leave a *turn-taking* signature (R < 1). This notebook asks the sharper question: **can a provider tell *how* a pool is being dispatched — and reconstruct it — from its own request log alone (key, arrival, end of stream)?**

Four signals, all computed without labels (library `fplib.py`):

| signal | what it measures | which dispatcher it exposes |
|---|---|---|
| **ring** | zero-lag high-pass occupancy correlation, keys ordered on a circle recovered by spectral seriation | one shared round-robin counter (`itertools.cycle` shared by workers) — also open-loop relays that round-robin, which R cannot see |
| **conc** | how concentrated each key's short-lag *end → next start* successors are | per-worker round-robin counters (key *i* is always followed by key *i+1*) |
| **coll** | same-key overlap relative to spreading the concurrent requests uniformly over the pool | least-busy / round-robin (avoid collisions) vs random / shuffle (≈ 1) |
| **R(τ)** | occupancy variance ratio at τ = 5, 20, 120 s and the full window | closed loops (R < 1); sticky / episodic key holding (R ≈ 1 at short τ, < 1 at long τ) |

plus a worker-count estimate **K = X · (E[D] + E[G])** (the interactive response-time law, N = X(R+Z)), where the client "think time" E[G] is read off the excess of *end → next start* lags.

**Pre-registered classifier and thresholds** were fixed in simulation (10/10 dispatchers correct on three seeds) before this run: ring z > 6 → shared round-robin (closed loop if R is also significantly < 1, otherwise open-loop relay); else conc z > 6 → per-worker round-robin; else if R or R20 is > 3 sd below the human null → sticky if (1−R5) < ½(1−R), least-busy if coll < 0.5, otherwise random/shuffle; R > 3 sd above → bunching; otherwise none.

**Fixes from Notebook 5.** Every population now runs in **its own process** (no shared event loop), the open-loop relays use an **absolute Poisson schedule**, and every request carries an `X-Req-Id` header so client-side and gateway-side times can be joined.

| population | dispatcher | K workers | keys | expected class |
|---|---|---|---|---|
| `hum` | 400 humans (two processes) | — | 400 | none |
| `bat` | 4 legitimate batch jobs (one key each) | — | 4 | (used in the heavy-benign control) |
| `P1_shuffle` | LiteLLM Router `simple-shuffle` | 6 | 40 | random / shuffle |
| `P2_least` | LiteLLM Router `least-busy` | 10 | 40 | **none** — the dry run showed LiteLLM breaks ties by list order, so the 10 workers settle on the first 10 keys and behave like 10 independent single-key clients (a known blind spot: pinned assignment gives R = 1) |
| `P3_rrshared` | OpenAI SDK, one shared `itertools.cycle` | 12 | 40 | shared round-robin (closed loop) |
| `P4_rrworker` | OpenAI SDK, one cycle per worker | 8 | 40 | per-worker round-robin |
| `P5_rrjitter` | shared cycle + 0–6 s random delay | 8 | 40 | shared round-robin (closed loop) |
| `P6_agents` | agents, one key per episode, tool calls | 6 | 32 | sticky / episodic |
| `P7_relay_random` | open-loop relay, Poisson 2/s, random key | — | 40 | none (undetectable by design) |
| `P8_relay_rr` | open-loop relay, Poisson 2/s, round-robin key | — | 40 | shared round-robin (open-loop relay) |
| `C_company` | legitimate company, LiteLLM shuffle | 4 | 10 | random / shuffle (designed false positive) |

**Run settings.** Accelerator **GPU T4 ×1**, Internet **On**. Runtime ≈ 75–90 min. Paste back the block between the PASTE markers.

In [ ]:
# Cell 1 — settings
import os, sys, json, time, subprocess, signal, glob
T_START = time.time()
NH = int(os.environ.get("NB6_HUMANS", 400))
CFG = dict(
    backend=os.environ.get("NB6_BACKEND", "vllm"),   # "vllm" = stop if vLLM cannot start; "auto" = fall back to the simulated server; "sim"
    model="Qwen/Qwen2.5-0.5B-Instruct",
    run_s=int(os.environ.get("NB6_RUN_S", 2400)), warmup_s=int(os.environ.get("NB6_WARMUP", 180)), seed=6,
    tok_med=96, tok_sig=0.6, client_gap_med=0.2,
    human=dict(gap_s=float(os.environ.get("NB6_HUMAN_GAP", 360)), mean_req=6, think_med_s=6.0, think_sig=0.9),
    agent=dict(mean_steps=8, tool_med_s=2.0, tok_med=64),
    pops={
        "hum_a": dict(kind="human", n=NH // 2, key_offset=0),
        "hum_b": dict(kind="human", n=NH - NH // 2, key_offset=NH // 2),
        "bat": dict(kind="batch", n=4),
        "P1_shuffle": dict(kind="litellm", strategy="simple-shuffle", K=6, M=40),
        "P2_least": dict(kind="litellm", strategy="least-busy", K=10, M=40),
        "P3_rrshared": dict(kind="rr_shared", K=12, M=40),
        "P4_rrworker": dict(kind="rr_worker", K=8, M=40),
        "P5_rrjitter": dict(kind="rr_shared", K=8, M=40, jitter=6.0),
        "P6_agents": dict(kind="agents", K=6, M=32),
        "P7_relay_random": dict(kind="open_loop", lam=2.0, policy="random", M=40),
        "P8_relay_rr": dict(kind="open_loop", lam=2.0, policy="rr", M=40),
        "C_company": dict(kind="litellm", strategy="simple-shuffle", K=4, M=10),
    },
    expected={
        "hum_random40": "none", "benign_36hum_4bat": "none",
        "P1_shuffle": "random / shuffle", "P2_least": "none",   # dry run: LiteLLM least-busy breaks ties by list order, so K workers settle on the first K keys (pinned)
        "P3_rrshared": "shared round-robin (closed loop)", "P4_rrworker": "per-worker round-robin",
        "P5_rrjitter": "shared round-robin (closed loop)", "P6_agents": "sticky / episodic",
        "P7_relay_random": "none", "P8_relay_rr": "shared round-robin (open-loop relay)", "C_company": "random / shuffle",
    },
    gateway="http://127.0.0.1:9000",
)
print(json.dumps({k: v for k, v in CFG.items() if k not in ("pops", "expected")}, indent=1))


In [ ]:
%%writefile occlib.py
"""
occlib — occupancy statistics for detecting hidden closed-loop worker pools behind many accounts.

Data model: a request log as three equal-length arrays
    acct : int   account index 0..A-1
    s    : float request arrival time (seconds)
    e    : float time the streamed response finished (seconds), e >= s
X_i(t) = number of account i's requests in flight; N(t) = sum_i X_i(t).

All statistics are exact time integrals of step functions (no time grid).
"""
import numpy as np

__version__ = "0.1.0"

# --------------------------------------------------------------------------------------------
# core: exact per-group integrals of X and X^2
# --------------------------------------------------------------------------------------------
def _group_integrals(gid, s, e, ngroups):
    """For intervals carrying a group id, return I1[g] = int X_g dt and I2[g] = int X_g^2 dt,
    where X_g(t) = number of the group's intervals covering t (half-open [s, e))."""
    I1 = np.zeros(ngroups); I2 = np.zeros(ngroups)
    n = len(s)
    if n == 0:
        return I1, I2
    t = np.concatenate([s, e])
    d = np.concatenate([np.ones(n), -np.ones(n)])
    g = np.concatenate([gid, gid])
    o = np.lexsort((d, t, g))          # by group, then time, then ends before starts
    t, d, g = t[o], d[o], g[o]
    x = np.cumsum(d)                    # each group nets to zero, so the level resets between groups
    dt = np.empty_like(t)
    dt[:-1] = t[1:] - t[:-1]
    dt[-1] = 0.0
    dt[:-1][g[1:] != g[:-1]] = 0.0
    I1 += np.bincount(g, weights=x * dt, minlength=ngroups)
    I2 += np.bincount(g, weights=x * x * dt, minlength=ngroups)
    return I1, I2


def _clip(acct, s, e, t0, t1):
    s = np.clip(np.asarray(s, float), t0, t1); e = np.clip(np.asarray(e, float), t0, t1)
    m = e > s
    return np.asarray(acct)[m], s[m], e[m]


def _split_blocks(acct, s, e, t0, tau, nb):
    """Split intervals at block boundaries t0 + k*tau. Returns (acct, block, s, e) pieces."""
    b0 = np.floor((s - t0) / tau).astype(np.int64)
    b1 = np.floor((np.nextafter(e, -np.inf) - t0) / tau).astype(np.int64)
    b0 = np.clip(b0, 0, nb - 1); b1 = np.clip(b1, 0, nb - 1)
    cnt = b1 - b0 + 1
    idx = np.repeat(np.arange(len(s)), cnt)
    off = np.arange(cnt.sum()) - np.repeat(np.cumsum(cnt) - cnt, cnt)
    blk = b0[idx] + off
    ps = np.maximum(s[idx], t0 + blk * tau)
    pe = np.minimum(e[idx], t0 + (blk + 1) * tau)
    m = pe > ps
    return acct[idx][m], blk[m], ps[m], pe[m]


def occupancy(acct, s, e, t0, t1, tau=None, n_accounts=None):
    """Occupancy moments over [t0, t1].
    tau=None -> full-window variances; tau=float -> mean over blocks of within-block variances
    (the 'high-pass' statistic). Returns dict with R, V_N, V_i (per account), mean_N, n_blocks."""
    acct = np.asarray(acct, np.int64)
    A = int(n_accounts if n_accounts is not None else (acct.max() + 1 if len(acct) else 0))
    if tau is None:
        tau = t1 - t0
    nb = int(np.floor((t1 - t0) / tau + 1e-9))
    if nb < 1:
        raise ValueError("window shorter than one block")
    t1 = t0 + nb * tau
    a, s, e = _clip(acct, s, e, t0, t1)
    a, b, s, e = _split_blocks(a, s, e, t0, tau, nb)
    # aggregate N, per block
    I1, I2 = _group_integrals(b, s, e, nb)
    VN_b = I2 / tau - (I1 / tau) ** 2
    # per account per block (compress group ids)
    key = a * nb + b
    uk, inv = np.unique(key, return_inverse=True)
    J1, J2 = _group_integrals(inv, s, e, len(uk))
    Vg = J2 / tau - (J1 / tau) ** 2
    V_i = np.bincount((uk // nb).astype(np.int64), weights=Vg, minlength=A) / nb
    V_N = VN_b.mean()
    sV = V_i.sum()
    return dict(R=V_N / sV if sV > 0 else np.nan, V_N=V_N, V_i=V_i, sumV=sV,
                mean_N=I1.sum() / (nb * tau), n_blocks=nb, tau=tau)


def R_full(acct, s, e, t0, t1):
    return occupancy(acct, s, e, t0, t1, None)["R"]


def R_hp(acct, s, e, t0, t1, tau=60.0):
    return occupancy(acct, s, e, t0, t1, tau)["R"]


def fano_khat(acct, s, e, t0, t1):
    """Fano factor of N and the implied effective number of hidden slots K = <N>^2/(<N>-Var N)."""
    o = occupancy(acct, s, e, t0, t1, None)
    F = o["V_N"] / o["mean_N"] if o["mean_N"] > 0 else np.nan
    p = 1 - F
    K = o["mean_N"] / p if p > 0 else np.inf
    return dict(F=F, p_hat=p, K_hat=K, mean_N=o["mean_N"], V_N=o["V_N"])


def baselines(acct, s, e, t0, t1):
    """Simple baselines: requests/hour, peak concurrency, Fano factor of N."""
    a, sc, ec = _clip(acct, s, e, t0, t1)
    n_req = int(((np.asarray(s) >= t0) & (np.asarray(s) < t1)).sum())
    t = np.concatenate([sc, ec]); d = np.concatenate([np.ones(len(sc)), -np.ones(len(ec))])
    o = np.lexsort((d, t)); peak = int(np.cumsum(d[o]).max()) if len(t) else 0
    f = fano_khat(acct, s, e, t0, t1)
    return dict(req_per_h=n_req / ((t1 - t0) / 3600), peak_N=peak, fano=f["F"])


# --------------------------------------------------------------------------------------------
# nulls
# --------------------------------------------------------------------------------------------
def circular_shift(acct, s, e, t0, t1, rng):
    """Shift each account's timeline by an independent uniform offset, wrapping around [t0, t1).
    Keeps each account's marginal behaviour and autocorrelation; destroys cross-account dependence."""
    acct = np.asarray(acct, np.int64); L = t1 - t0
    a, s, e = _clip(acct, s, e, t0, t1)
    A = int(acct.max() + 1) if len(acct) else 0
    off = rng.uniform(0, L, A)[a]
    s2 = t0 + np.mod(s - t0 + off, L); e2 = s2 + (e - s)
    wrap = e2 > t1
    a_out = np.concatenate([a, a[wrap]])
    s_out = np.concatenate([s2, np.full(wrap.sum(), t0)])
    e_out = np.concatenate([np.minimum(e2, t1), t0 + (e2[wrap] - t1)])
    return a_out, s_out, e_out


def surrogate_null(acct, s, e, t0, t1, tau, n_sur, rng):
    """Distribution of R (tau=None) or R_tau under independent circular shifts."""
    return np.array([occupancy(*circular_shift(acct, s, e, t0, t1, rng), t0, t1, tau)["R"] for _ in range(n_sur)])


# --------------------------------------------------------------------------------------------
# pool growth: correlation of each candidate's occupancy with a pool's aggregate occupancy
# --------------------------------------------------------------------------------------------
def _cum_integral(s, e, t0, t1):
    """Knots and running integral F(t) = int_t0^t N(u) du for the superposition of intervals."""
    t = np.concatenate([[t0], s, e, [t1]])
    d = np.concatenate([[0.0], np.ones(len(s)), -np.ones(len(e)), [0.0]])
    o = np.lexsort((d, t)); t, d = t[o], d[o]
    x = np.cumsum(d)
    F = np.concatenate([[0.0], np.cumsum(x[:-1] * np.diff(t))])
    return t, F


def corr_with_pool(pool, cand, t0, t1, tau=None, n_cand=None):
    """pool=(acct,s,e) of pool members; cand=(acct,s,e) of candidates (acct = candidate index 0..C-1).
    Returns per-candidate correlation of X_j with N_pool (within-block if tau is set). Linear in traffic."""
    pa, ps, pe = _clip(*pool, t0, t1)
    ca, cs, ce = _clip(*cand, t0, t1)
    C = int(n_cand if n_cand is not None else np.asarray(cand[0]).max() + 1)
    if tau is None:
        tau = t1 - t0
    nb = int(np.floor((t1 - t0) / tau + 1e-9)); t1 = t0 + nb * tau
    # pool per-block mean and within-block variance
    _, pb, pps, ppe = _split_blocks(pa, np.clip(ps, t0, t1), np.clip(pe, t0, t1), t0, tau, nb)
    I1, I2 = _group_integrals(pb, pps, ppe, nb)
    Nbar = I1 / tau; VN = (I2 / tau - Nbar ** 2).mean()
    kn, F = _cum_integral(np.clip(ps, t0, t1), np.clip(pe, t0, t1), t0, t1)
    # candidate pieces
    ca2, cb, cs2, ce2 = _split_blocks(ca, np.clip(cs, t0, t1), np.clip(ce, t0, t1), t0, tau, nb)
    XN = np.interp(ce2, kn, F) - np.interp(cs2, kn, F)       # int over piece of N_pool
    key = ca2 * nb + cb
    uk, inv = np.unique(key, return_inverse=True)
    A = np.bincount(inv, weights=XN)                          # int X_j N over block
    B = np.bincount(inv, weights=ce2 - cs2)                   # int X_j over block
    jj = uk // nb; bb = uk % nb
    cov = np.bincount(jj, weights=A / tau - (B / tau) * Nbar[bb], minlength=C) / nb
    Vj = occupancy(ca, cs, ce, t0, t1, None if nb == 1 else tau, n_accounts=C)["V_i"]
    with np.errstate(invalid="ignore", divide="ignore"):
        return cov / np.sqrt(Vj * VN)


# --------------------------------------------------------------------------------------------
# generators (simulation)
# --------------------------------------------------------------------------------------------
DUR_MED, DUR_SIG = 8.0, 0.6       # streamed response duration, log-normal
GAP_MED, GAP_SIG = 0.4, 0.5       # client processing gap, log-normal
E_DUR = DUR_MED * np.exp(DUR_SIG ** 2 / 2)
E_GAP = GAP_MED * np.exp(GAP_SIG ** 2 / 2)


def duty(J):
    return E_DUR / (E_DUR + E_GAP + J / 2.0)


def theory_R(p, M):
    return (1 - p) / (1 - p / M)


def _pack(rows):
    if not rows:
        return np.zeros(0, np.int64), np.zeros(0), np.zeros(0)
    a, s, e = map(np.asarray, zip(*rows))
    return a.astype(np.int64), s.astype(float), e.astype(float)


def benign_humans(rng, n_accounts, T, session_rate=1 / 7200, mean_req=6, think_med=25.0, think_sig=0.9,
                  diurnal=0.0, offset=0):
    """Human-like accounts: Poisson session starts, geometric #requests, log-normal think time.
    diurnal in [0,1): amplitude of a 24 h sinusoidal rate modulation (common to all accounts)."""
    rows = []
    for i in range(n_accounts):
        t = rng.exponential(1 / session_rate)
        while t < T:
            if diurnal > 0 and rng.random() > (1 + diurnal * np.sin(2 * np.pi * t / 86400)) / (1 + diurnal):
                t += rng.exponential(1 / session_rate); continue
            tt = t
            for _ in range(rng.geometric(1 / mean_req)):
                if tt > T: break
                d = rng.lognormal(np.log(DUR_MED), DUR_SIG); rows.append((offset + i, tt, tt + d))
                tt += d + rng.lognormal(np.log(think_med), think_sig)
            t = tt + rng.exponential(1 / session_rate)
    return _pack(rows)


def closed_loop_pool(rng, T, K, M, J=0.0, assign="random", stint=None, K_bar=None, epoch=None, offset=0):
    """K closed-loop slots over M accounts. Each slot: send, wait for stream end, gap, uniform(0,J) delay.
    assign: 'random' (per request), 'pinned' (slot w -> account w % M), 'stint' (hold an account for `stint` s).
    K_bar/epoch: if set, the number of active slots is redrawn ~ Poisson(K_bar) (capped at K) every `epoch` s."""
    rows = []
    Kact = None
    if K_bar is not None:
        Kact = np.minimum(rng.poisson(K_bar, int(T // epoch) + 2), K)
    held = {}
    for w in range(K):
        t = rng.uniform(0, 5)
        while t < T:
            if Kact is not None:
                ei = int(t // epoch)
                if w >= Kact[ei]:
                    t = (ei + 1) * epoch + rng.uniform(0, 1); continue
            if assign == "pinned":
                a = w % M
            elif assign == "stint":
                k = (w, int(t // stint))
                if k not in held: held[k] = rng.integers(M)
                a = held[k]
            else:
                a = rng.integers(M)
            d = rng.lognormal(np.log(DUR_MED), DUR_SIG)
            rows.append((offset + a, t, t + d))
            t += d + rng.lognormal(np.log(GAP_MED), GAP_SIG) + (rng.uniform(0, J) if J > 0 else 0.0)
    return _pack(rows)


def open_loop_relay(rng, T, M, lam, policy="random", offset=0):
    """Open-loop Poisson arrivals (rate lam/s) dispatched over M accounts: random | rr | least."""
    import heapq
    rows = []; inflight = np.zeros(M, int); heap = []; t = 0.0; rr = 0
    while True:
        t += rng.exponential(1 / lam)
        if t > T: break
        while heap and heap[0][0] <= t:
            _, a = heapq.heappop(heap); inflight[a] -= 1
        if policy == "random": a = int(rng.integers(M))
        elif policy == "rr": a = rr % M; rr += 1
        else:
            m = inflight.min(); a = int(rng.choice(np.flatnonzero(inflight == m)))
        d = rng.lognormal(np.log(DUR_MED), DUR_SIG)
        rows.append((offset + a, t, t + d)); inflight[a] += 1; heapq.heappush(heap, (t + d, a))
    return _pack(rows)


def concat(*logs):
    return tuple(np.concatenate(x) for x in zip(*logs))


def relabel(acct):
    """Map arbitrary account ids to 0..A-1."""
    u, inv = np.unique(acct, return_inverse=True)
    return inv.astype(np.int64), u


In [ ]:
%%writefile fplib.py
"""fplib — dispatcher fingerprinting from per-key request logs (start, end) only.

Signals (all computed without labels):
  ring     zero-lag high-pass occupancy correlation arranged on a recovered circle (shared round-robin counter)
  conc     concentration of each key's short-lag end->start successors (per-worker round-robin counters)
  coll     same-key overlap relative to uniform spreading of concurrent requests (least-busy / round-robin avoid collisions)
  R(tau)   occupancy variance ratio at several timescales (closed-loop turn-taking, sticky/episodic key holding)
  K_little worker count from the interactive response-time law, think time read off the successor-lag excess
"""
import numpy as np

__version__ = "fp-0.3"


# ---------------------------------------------------------------- occupancy on a grid
def occupancy_matrix(a, s, e, M, t0, t1, dt=0.05):
    n = int((t1 - t0) / dt)
    i0 = np.clip(np.floor((s - t0) / dt).astype(np.int64), 0, n)
    i1 = np.clip(np.floor((e - t0) / dt).astype(np.int64), 0, n)
    D = np.zeros((M, n + 1), np.float32)
    np.add.at(D, (a, i0), 1.0); np.add.at(D, (a, i1), -1.0)
    return np.cumsum(D, 1)[:, :n]


def highpass(X, block):
    M, n = X.shape; nb = n // block
    Y = X[:, : nb * block].reshape(M, nb, block)
    return (Y - Y.mean(2, keepdims=True)).reshape(M, nb * block)


def corr_matrix(X):
    C = np.cov(X.astype(np.float64)); d = np.sqrt(np.clip(np.diag(C), 0, None))
    with np.errstate(invalid="ignore", divide="ignore"):
        R = C / np.outer(d, d)
    R[~np.isfinite(R)] = 0.0
    return R


# ---------------------------------------------------------------- circular seriation
def circular_seriation(C):
    """Order keys around a circle from a similarity matrix (angle in the plane of two leading eigenvectors)."""
    S = np.clip(C, 0, None).copy(); np.fill_diagonal(S, 0)
    w, V = np.linalg.eigh(S)
    best = None
    for i, j in [(1, 2), (2, 3), (1, 3)]:
        x, y = V[:, -i], V[:, -j]
        r = np.hypot(x, y)
        score = r.std() / (r.mean() + 1e-12)          # points of a ring embed at constant radius
        if best is None or score < best[0]:
            best = (score, np.arctan2(y, x))
    return np.argsort(best[1])


def ring_score(C, order):
    """mean correlation between circular neighbours in the recovered order minus mean off-diagonal correlation"""
    M = len(order)
    nb = np.mean([C[order[i], order[(i + 1) % M]] for i in range(M)])
    off = C[~np.eye(M, dtype=bool)].mean()
    return float(nb - off)


def neighbour_accuracy(order, true_pos):
    """fraction of recovered neighbours that are true ring neighbours; true_pos[k] = key k's true ring position"""
    M = len(order); p = np.asarray(true_pos)[order]
    return float(np.mean([((p[i] - p[(i + 1) % M]) % M) in (1, M - 1) for i in range(M)]))


# ---------------------------------------------------------------- end -> start successors
def handoff_counts(a, s, e, M, lag_lo=0.0, lag_hi=1.0):
    """H[i, j] = number of (end on key i, start on key j within (lag_lo, lag_hi] s after that end)"""
    o = np.argsort(s, kind="stable"); ss, sa = s[o], a[o]
    lo = np.searchsorted(ss, e + lag_lo, "right"); hi = np.searchsorted(ss, e + lag_hi, "right")
    cnt = hi - lo
    H = np.zeros((M, M))
    if cnt.sum() == 0:
        return H
    src = np.repeat(a, cnt)
    idx = np.repeat(lo - np.r_[0, np.cumsum(cnt)[:-1]], cnt) + np.arange(cnt.sum())
    np.add.at(H, (src, sa[idx]), 1)
    return H


def handoff_excess(a, s, e, M, t0, t1, lag_hi=1.0):
    """observed minus expected successor counts (expected: key j's starts spread uniformly over time)"""
    H = handoff_counts(a, s, e, M, 0.0, lag_hi)
    ends = np.bincount(a, minlength=M).astype(float); rate = ends / (t1 - t0)
    Ex = np.outer(ends, rate) * lag_hi
    return H, Ex


def successor_concentration(a, s, e, M, t0, t1, lag_hi=1.0, min_rows=5):
    H, Ex = handoff_excess(a, s, e, M, t0, t1, lag_hi)
    X = np.clip(H - Ex, 0, None); np.fill_diagonal(X, 0)
    rows = X.sum(1); ok = rows >= min_rows
    if ok.sum() < 3:
        return float("nan"), X
    return float(np.mean(X[ok].max(1) / rows[ok])), X


def ring_offsets(X, order):
    """distribution of successor offsets measured along the recovered circle (direction chosen by mass)"""
    M = len(order); pos = np.empty(M, int); pos[order] = np.arange(M)
    off = np.zeros(M)
    I, J = np.nonzero(X)
    np.add.at(off, (pos[J] - pos[I]) % M, X[I, J])
    h = M // 2
    if off[h + 1:].sum() > off[1:h].sum():
        off = np.r_[off[0], off[1:][::-1]]
    return off


def K_ring(X, order):
    off = ring_offsets(X, order); M = len(order); h = M // 2
    w = off[1:h]
    return float((np.arange(1, h) * w).sum() / w.sum()) if w.sum() > 0 else float("nan")


# ---------------------------------------------------------------- collisions
def collision_ratio(a, s, e, M, t0, t1):
    """time-integrated same-key concurrent pairs / expected if concurrent requests were spread uniformly over M keys"""
    ev = np.r_[s, e]; dlt = np.r_[np.ones(len(s), int), -np.ones(len(e), int)]; kk = np.r_[a, a]
    o = np.lexsort((dlt, ev)); ev, dlt, kk = ev[o], dlt[o], kk[o]
    X = np.zeros(M, int); N = 0; pairs = 0; obs = exp_ = 0.0; last = t0
    for t, d, k in zip(ev.tolist(), dlt.tolist(), kk.tolist()):
        span = min(t, t1) - max(last, t0)
        if span > 0:
            obs += pairs * span; exp_ += N * (N - 1) / 2.0 / M * span
        last = t
        if d > 0: pairs += X[k]
        else: pairs -= X[k] - 1
        X[k] += d; N += d
    return float(obs / exp_) if exp_ > 0 else float("nan")


# ---------------------------------------------------------------- surrogates
def shift_keys(a, s, e, t0, t1, rng, M):
    """independent circular time shift per key (keeps each key's own pattern, destroys cross-key timing)"""
    L = t1 - t0; off = rng.uniform(0, L, M)[a]
    s2 = t0 + np.mod(s - t0 + off, L)
    return a, s2, s2 + (e - s)


# ---------------------------------------------------------------- worker count from the interactive response-time law
LAG_EDGES = np.r_[0, 0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8, 12, 16]


def excess_lag_hist(s, e, t0, t1, edges=LAG_EDGES):
    """pool-level histogram of (end -> any later start in the pool) lags minus its expectation if starts were uniform in time"""
    o = np.sort(s); lam = len(s) / (t1 - t0)
    return np.array([(np.searchsorted(o, e + hi, "right") - np.searchsorted(o, e + lo, "right")).sum() - len(e) * lam * (hi - lo)
                     for lo, hi in zip(edges[:-1], edges[1:])])


def K_little(s, e, t0, t1, edges=LAG_EDGES):
    """K = X * (E[D] + E[G]) (Denning-Buzen interactive response time law, N = X (R + Z)) with throughput X, mean response
    E[D] and the client gap ('think time') E[G] read off the excess successor-lag histogram. Valid for always-busy closed loops."""
    h = np.clip(excess_lag_hist(s, e, t0, t1, edges), 0, None); mid = (edges[:-1] + edges[1:]) / 2
    EG = float((h * mid).sum() / h.sum()) if h.sum() > 0 else float("nan")
    lam = len(s) / (t1 - t0)
    return lam * (float(np.mean(e - s)) + EG), EG


# ---------------------------------------------------------------- full fingerprint of one pool
def best_lag_window(a, s, e, M, t0, t1, windows=(0.5, 1.0, 2.0, 4.0, 8.0)):
    """successor window with the largest cross-key excess z (sharpest view of the closed-loop hand-off)"""
    off = ~np.eye(M, dtype=bool); best = (windows[0], -np.inf)
    for w in windows:
        H, Ex = handoff_excess(a, s, e, M, t0, t1, w)
        z = (H[off].sum() - Ex[off].sum()) / np.sqrt(Ex[off].sum() + 1)
        if z > best[1]: best = (w, float(z))
    return best


def fingerprint(a, s, e, M, t0, t1, rng, n_sur=40, dt=0.05, hp_block_s=20.0, lag_hi=None, occupancy_fn=None):
    """a: keys 0..M-1 (requests overlapping [t0, t1)). Returns (features, recovered circular order)."""
    m = (e > t0) & (s < t1); a, s, e = a[m], s[m], e[m]
    out = dict(n_req=int(len(a)), n_keys=int(M))
    if lag_hi is None:
        lag_hi, out["handoff_z"] = best_lag_window(a, s, e, M, t0, t1)
    out["lag_win"] = lag_hi
    if occupancy_fn is not None:
        for tau in (None, 5.0, 20.0, 120.0):
            if tau is None or (t1 - t0) >= 2 * tau:
                out["R" if tau is None else f"R{int(tau)}"] = float(occupancy_fn(a, s, e, t0, t1, tau, n_accounts=M)["R"])
    out["coll"] = collision_ratio(a, s, e, M, t0, t1)
    out["K_little"], out["think_s"] = K_little(s, e, t0, t1)
    def ring_of(a_, s_, e_):
        C = corr_matrix(highpass(occupancy_matrix(a_, s_, e_, M, t0, t1, dt), int(hp_block_s / dt)))
        o = circular_seriation(C); return ring_score(C, o), o
    out["ring"], order = ring_of(a, s, e)
    out["conc"], X = successor_concentration(a, s, e, M, t0, t1, lag_hi)
    if n_sur == 0:
        return out, order
    sr, sc = [], []
    for _ in range(n_sur):
        aa, ss, ee = shift_keys(a, s, e, t0, t1, rng, M)
        sr.append(ring_of(aa, ss, ee)[0]); sc.append(successor_concentration(aa, ss, ee, M, t0, t1, lag_hi)[0])
    sr, sc = np.array(sr), np.array(sc)
    out["ring_p"] = float((1 + (sr >= out["ring"]).sum()) / (n_sur + 1))
    out["ring_z"] = float((out["ring"] - sr.mean()) / (sr.std() + 1e-12))
    sc = sc[np.isfinite(sc)]
    ok = len(sc) > 2 and np.isfinite(out["conc"])
    out["conc_z"] = float((out["conc"] - sc.mean()) / (sc.std() + 1e-12)) if ok else float("nan")
    out["conc_p"] = float((1 + (sc >= out["conc"]).sum()) / (len(sc) + 1)) if ok else float("nan")
    return out, order


# ---------------------------------------------------------------- pre-registered classifier
CLASSES = ["none", "shared round-robin (closed loop)", "shared round-robin (open-loop relay)", "per-worker round-robin",
           "random / shuffle", "least-busy", "sticky / episodic", "bunching"]


def classify(f, zR, z_thr=3.0, ring_z_thr=6.0, conc_z_thr=6.0):
    """f: fingerprint dict; zR: dict of z-scores of R-type statistics against the human null (positive = turn-taking).
    Thresholds are fixed in advance from simulation (fp/explore*.py), not tuned on the testbed."""
    z_full, z_short = zR.get("R", 0.0), zR.get("R5", 0.0)
    turn = max(z_full, zR.get("R20", 0.0)) > z_thr
    if f["ring_z"] > ring_z_thr:
        return CLASSES[1] if turn else CLASSES[2]
    if np.isfinite(f.get("conc_z", np.nan)) and f["conc_z"] > conc_z_thr:
        return CLASSES[3]
    if turn:
        r_full, r5 = f.get("R", 1.0), f.get("R5", 1.0)
        if (1 - r5) < 0.5 * (1 - r_full):
            return CLASSES[6]
        return CLASSES[5] if f["coll"] < 0.5 else CLASSES[4]
    if min(z_full, zR.get("R20", 0.0)) < -z_thr:
        return CLASSES[7]
    return CLASSES[0]


# ---------------------------------------------------------------- site-wide blind discovery
def site_handoff_graph(a, s, e, N, t0, t1, lag_hi=1.0, q=1e-3, min_count=4):
    """Symmetric key-pair successor excess with exact Poisson tails, Benjamini-Hochberg over all pairs.
    Returns component labels (-1 = not in any component) and the edge list."""
    from scipy.stats import poisson
    H, Ex = handoff_excess(a, s, e, N, t0, t1, lag_hi)
    np.fill_diagonal(H, 0); np.fill_diagonal(Ex, 0)
    Hs, Es = H + H.T, Ex + Ex.T
    iu = np.triu_indices(N, 1)
    obs, exp_ = Hs[iu], Es[iu]
    p = poisson.sf(obs - 1, np.maximum(exp_, 1e-12))
    p[obs < min_count] = 1.0
    m = len(p); o = np.argsort(p); thr = q * np.arange(1, m + 1) / m
    passed = np.zeros(m, bool)
    k = np.nonzero(p[o] <= thr)[0]
    if len(k): passed[o[: k.max() + 1]] = True
    I, J = iu[0][passed], iu[1][passed]
    parent = np.arange(N)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i, j in zip(I, J):
        ri, rj = find(i), find(j)
        if ri != rj: parent[ri] = rj
    roots = np.array([find(i) for i in range(N)])
    deg = np.bincount(np.r_[I, J], minlength=N)
    lab = np.where(deg > 0, roots, -1)
    return lab, list(zip(I.tolist(), J.tolist()))


In [ ]:
%%writefile simserver.py
"""Simulated OpenAI-compatible streaming server (fallback when no GPU/vLLM).
Latency model: requests beyond `capacity` wait in a FIFO queue; per-token time grows with the number of active
sequences (batching interference): tpot = base_tpot * (1 + slowdown * active / capacity)."""
import asyncio, json, time, argparse, random
from aiohttp import web

ap = argparse.ArgumentParser()
ap.add_argument("--port", type=int, default=8000)
ap.add_argument("--capacity", type=int, default=128)
ap.add_argument("--base_tpot", type=float, default=0.02)
ap.add_argument("--slowdown", type=float, default=1.0)
ap.add_argument("--ttft", type=float, default=0.08)
args = ap.parse_args()
sem = asyncio.Semaphore(args.capacity)
active = 0

async def chat(request):
    global active
    body = await request.json()
    n = int(body.get("max_tokens") or 64)
    resp = web.StreamResponse(status=200, headers={"Content-Type": "text/event-stream"})
    await resp.prepare(request)
    async with sem:
        active += 1
        try:
            await asyncio.sleep(args.ttft * random.uniform(0.8, 1.2))
            for i in range(n):
                chunk = {"id": "x", "object": "chat.completion.chunk", "created": int(time.time()), "model": body.get("model", "sim"),
                         "choices": [{"index": 0, "delta": {"content": f"w{random.randint(0, 9999)} "}, "finish_reason": None}]}
                await resp.write(f"data: {json.dumps(chunk)}\n\n".encode())
                await asyncio.sleep(args.base_tpot * (1 + args.slowdown * active / args.capacity))
            fin = {"id": "x", "object": "chat.completion.chunk", "created": int(time.time()), "model": body.get("model", "sim"),
                   "choices": [{"index": 0, "delta": {}, "finish_reason": "length"}]}
            await resp.write(f"data: {json.dumps(fin)}\n\n".encode())
            await resp.write(b"data: [DONE]\n\n")
        finally:
            active -= 1
    await resp.write_eof()
    return resp

async def models(request):
    return web.json_response({"object": "list", "data": [{"id": "sim", "object": "model"}]})

async def health(request):
    return web.Response(text="ok")

app = web.Application()
app.router.add_post("/v1/chat/completions", chat)
app.router.add_get("/v1/models", models)
app.router.add_get("/health", health)
web.run_app(app, port=args.port, print=None, access_log=None)


In [ ]:
%%writefile gateway.py
"""Logging API gateway: authenticates by API key, forwards to the model server, streams the answer back,
and logs one line per request: key, arrival time, first byte, stream end, status, bytes.
This is what a provider sees: arrival -> end of stream at its own edge (queueing included)."""
import asyncio, time, argparse
import aiohttp
from aiohttp import web

ap = argparse.ArgumentParser()
ap.add_argument("--port", type=int, default=9000)
ap.add_argument("--backend", default="http://127.0.0.1:8000")
ap.add_argument("--log", default="gateway_log.csv")
args = ap.parse_args()
logf = open(args.log, "a", buffering=1)
if logf.tell() == 0:
    logf.write("key,rid,t_arrival,t_first,t_end,status,nbytes\n")
session = None

async def on_start(app):
    global session
    session = aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=0), timeout=aiohttp.ClientTimeout(total=None))

async def on_stop(app):
    await session.close(); logf.close()

async def proxy(request):
    key = request.headers.get("Authorization", "").replace("Bearer ", "").strip() or "anon"
    rid = request.headers.get("X-Req-Id", "")
    t_arr = time.time(); t_first = None; nbytes = 0; status = 0
    body = await request.read()
    try:
        async with session.post(args.backend + request.path, data=body, headers={"Content-Type": "application/json"}) as r:
            status = r.status
            resp = web.StreamResponse(status=r.status, headers={"Content-Type": r.headers.get("Content-Type", "text/event-stream")})
            await resp.prepare(request)
            async for chunk in r.content.iter_any():
                if t_first is None:
                    t_first = time.time()
                await resp.write(chunk); nbytes += len(chunk)
            await resp.write_eof()
    except (ConnectionResetError, aiohttp.ClientError, asyncio.CancelledError, RuntimeError):
        status = 499                                   # client went away or backend failed mid-stream
        resp = web.Response(status=499)
    finally:
        logf.write(f"{key},{rid},{t_arr:.6f},{(t_first or 0):.6f},{time.time():.6f},{status},{nbytes}\n")
    return resp

async def passthrough_get(request):
    async with session.get(args.backend + request.path) as r:
        return web.Response(status=r.status, body=await r.read(), content_type="application/json")

app = web.Application(client_max_size=10 * 1024 ** 2)
app.on_startup.append(on_start); app.on_cleanup.append(on_stop)
app.router.add_post("/v1/chat/completions", proxy)
app.router.add_get("/v1/models", passthrough_get)
web.run_app(app, port=args.port, print=None, access_log=None)


In [ ]:
%%writefile driver6.py
"""One client population per process (no shared event loop between populations).
usage: python driver6.py cfg.json POP
Writes labels_POP.json (key -> POP, ring position) and clientlog_POP.csv (rid, key, t_sched, t_send, t_done, ok).
Every request carries an X-Req-Id header so client-side and gateway-side timestamps can be joined."""
import asyncio, json, sys, time, itertools, random, math, logging, uuid
import numpy as np
from openai import AsyncOpenAI

cfg = json.load(open(sys.argv[1])); POP = sys.argv[2]; P = cfg["pops"][POP]
seed = cfg["seed"] * 1000 + sorted(cfg["pops"]).index(POP)
rng = np.random.default_rng(seed); random.seed(seed)
BASE = cfg["gateway"] + "/v1"; MODEL = cfg["model"]
T0 = cfg["t_start"]; T_END = T0 + cfg["run_s"]
labels = {}; stats = {"ok": 0, "err": 0}; err_types = {}; _clients = {}
clog = open(f"clientlog_{POP}.csv", "w", buffering=1); clog.write("rid,key,t_sched,t_send,t_done,ok\n")

def keys(n, prefix=None):
    prefix = prefix or POP
    ks = [f"{prefix}-{i:03d}" for i in range(P.get("key_offset", 0), P.get("key_offset", 0) + n)]
    for i, k in enumerate(ks): labels[k] = dict(pop=POP, pos=i)
    return ks

def client(key):
    if key not in _clients:
        _clients[key] = AsyncOpenAI(api_key=key, base_url=BASE, max_retries=0, timeout=300)
    return _clients[key]

def ntok(med=None):
    return int(np.clip(rng.lognormal(math.log(med or cfg["tok_med"]), cfg["tok_sig"]), 16, 512))

def prompt():
    return [{"role": "user", "content": f"Write a short note about topic {random.randint(0, 10**6)}."}]

def note_err(ex):
    stats["err"] += 1
    k = f"{POP}:{type(ex).__name__}:{str(ex)[:80]}"; err_types[k] = err_types.get(k, 0) + 1

async def call(key, n, t_sched=None):
    rid = uuid.uuid4().hex[:16]; t_send = time.time(); ok = 0
    try:
        s = await client(key).chat.completions.create(model=MODEL, messages=prompt(), max_tokens=n, stream=True,
                                                      extra_body={"ignore_eos": True}, extra_headers={"X-Req-Id": rid})
        async for _ in s:
            pass
        stats["ok"] += 1; ok = 1
    except Exception as ex:
        note_err(ex); await asyncio.sleep(0.5)
    clog.write(f"{rid},{key},{t_sched or t_send:.6f},{t_send:.6f},{time.time():.6f},{ok}\n")

def alive():
    return time.time() < T_END

async def start_delay(hi):
    await asyncio.sleep(max(0.0, T0 - time.time()) + rng.uniform(0, hi))

def gap():
    return rng.lognormal(math.log(cfg["client_gap_med"]), 0.5)

# ---------------------------------------------------------------- benign
async def human(key):
    h = cfg["human"]
    await start_delay(0); await asyncio.sleep(rng.exponential(h["gap_s"]))
    while alive():
        for _ in range(rng.geometric(1 / h["mean_req"])):
            if not alive(): return
            await call(key, ntok())
            await asyncio.sleep(rng.lognormal(math.log(h["think_med_s"]), h["think_sig"]))
        await asyncio.sleep(rng.exponential(h["gap_s"]))

async def batch_client(key):
    await start_delay(5)
    while alive():
        await call(key, ntok()); await asyncio.sleep(gap())

# ---------------------------------------------------------------- dispatchers
async def rr_shared(ks, K, jitter=0.0):
    """K workers share ONE itertools.cycle over the keys (the common 'rotate keys' snippet)"""
    cyc = itertools.cycle(ks)
    async def worker():
        await start_delay(2)
        while alive():
            await call(next(cyc), ntok()); await asyncio.sleep(gap() + (rng.uniform(0, jitter) if jitter else 0.0))
    await asyncio.gather(*[worker() for _ in range(K)])

async def rr_per_worker(ks, K):
    """each worker has its OWN cycle, started at a random key"""
    async def worker():
        i = int(rng.integers(len(ks)))
        await start_delay(2)
        while alive():
            await call(ks[i % len(ks)], ntok()); i += 1; await asyncio.sleep(gap())
    await asyncio.gather(*[worker() for _ in range(K)])

async def litellm_router(ks, K, strategy):
    import litellm
    from litellm import Router
    litellm.suppress_debug_info = True
    if hasattr(litellm, "REPEATED_STREAMING_CHUNK_LIMIT"):
        litellm.REPEATED_STREAMING_CHUNK_LIMIT = 10 ** 9
    for n in ("LiteLLM", "LiteLLM Router", "LiteLLM Proxy"): logging.getLogger(n).setLevel(logging.ERROR)
    router = Router(model_list=[{"model_name": "teacher", "litellm_params": {"model": f"openai/{MODEL}", "api_base": BASE, "api_key": k}}
                                for k in ks], routing_strategy=strategy, num_retries=0)
    async def worker():
        await start_delay(2)
        while alive():
            rid = uuid.uuid4().hex[:16]; t_send = time.time(); ok = 0
            try:
                r = await router.acompletion(model="teacher", messages=prompt(), max_tokens=ntok(), stream=True,
                                             extra_body={"ignore_eos": True}, extra_headers={"X-Req-Id": rid})
                async for _ in r:
                    pass
                stats["ok"] += 1; ok = 1
            except Exception as ex:
                note_err(ex); await asyncio.sleep(0.5)
            clog.write(f"{rid},,{t_send:.6f},{t_send:.6f},{time.time():.6f},{ok}\n")
            await asyncio.sleep(gap())
    await asyncio.gather(*[worker() for _ in range(K)])

async def agent_fleet(ks, n_agents):
    a = cfg["agent"]
    async def agent():
        await start_delay(5)
        while alive():
            key = ks[rng.integers(len(ks))]
            for _ in range(rng.geometric(1 / a["mean_steps"])):
                if not alive(): return
                await call(key, ntok(a["tok_med"]))
                await asyncio.sleep(rng.lognormal(math.log(a["tool_med_s"]), 0.6))
            await asyncio.sleep(rng.lognormal(math.log(1.0), 0.5))
    await asyncio.gather(*[agent() for _ in range(n_agents)])

async def open_loop(ks, lam, policy):
    """Relay of independent end users: Poisson arrivals on an ABSOLUTE schedule (no drift from event-loop lag)."""
    tasks = set(); t = T0; rr = 0
    while True:
        t += rng.exponential(1 / lam)
        if t >= T_END: break
        await asyncio.sleep(max(0.0, t - time.time()))
        if policy == "rr": k = ks[rr % len(ks)]; rr += 1
        else: k = ks[rng.integers(len(ks))]
        task = asyncio.create_task(call(k, ntok(), t_sched=t)); tasks.add(task); task.add_done_callback(tasks.discard)
    if tasks:
        await asyncio.wait(tasks, timeout=120)

async def main():
    kind = P["kind"]
    if kind == "human":
        jobs = [human(k) for k in keys(P["n"], prefix="hum")]
    elif kind == "batch":
        jobs = [batch_client(k) for k in keys(P["n"], prefix="bat")]
    elif kind == "rr_shared":
        jobs = [rr_shared(keys(P["M"]), P["K"], P.get("jitter", 0.0))]
    elif kind == "rr_worker":
        jobs = [rr_per_worker(keys(P["M"]), P["K"])]
    elif kind == "litellm":
        jobs = [litellm_router(keys(P["M"]), P["K"], P["strategy"])]
    elif kind == "agents":
        jobs = [agent_fleet(keys(P["M"]), P["K"])]
    elif kind == "open_loop":
        jobs = [open_loop(keys(P["M"]), P["lam"], P["policy"])]
    else:
        raise ValueError(kind)
    json.dump(labels, open(f"labels_{POP}.json", "w"))
    async def progress():
        while alive():
            await asyncio.sleep(120)
            print(f"[{POP}] t={time.time() - T0:.0f}s ok={stats['ok']} err={stats['err']}", flush=True)
    await asyncio.gather(*jobs, progress())
    print(f"[{POP}] done ok={stats['ok']} err={stats['err']} errors={json.dumps(dict(sorted(err_types.items(), key=lambda kv: -kv[1])[:4]))}", flush=True)
    json.dump(dict(stats=stats, errors=err_types), open(f"driverstats_{POP}.json", "w"))

asyncio.run(main())


In [ ]:
# Cell 4 — client libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp", "openai", "litellm"], check=False)
import importlib; importlib.invalidate_caches()
import aiohttp, openai
print("aiohttp", aiohttp.__version__, "openai", openai.__version__)


In [ ]:
# Cell 5 — model server: vLLM on the GPU if possible, otherwise the simulated server
import urllib.request
def wait_http(url, timeout_s, proc=None):
    t = time.time()
    while time.time() - t < timeout_s:
        if proc is not None and proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen(url, timeout=5); return True
        except Exception:
            time.sleep(5)
    return False

def tail(path, n=25):
    try:
        return "".join(open(path, errors="ignore").readlines()[-n:])
    except Exception:
        return ""

BACKEND = {"kind": None, "attempts": []}
def gpu_present():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False

def vllm_import_check(py):
    r = subprocess.run([py, "-c", "import torch, vllm; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())"],
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-1200:]

def launch_vllm(py, extra_env, extra_args):
    env = dict(os.environ, **extra_env)
    log = open("vllm.log", "w")
    p = subprocess.Popen([py, "-m", "vllm.entrypoints.openai.api_server", "--model", CFG["model"], "--dtype", "half",
                          "--max-model-len", "2048", "--gpu-memory-utilization", "0.85", "--max-num-seqs", "128", "--port", "8000"] + extra_args,
                         stdout=log, stderr=subprocess.STDOUT, env=env)
    ok = wait_http("http://127.0.0.1:8000/v1/models", 900, p)
    if not ok:
        p.kill(); time.sleep(5)
    return ok, p

if CFG["backend"] in ("auto", "vllm") and gpu_present():
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
    pythons = []
    # Route 1: Kaggle's own Python. vLLM upgrades torch; Kaggle's torchaudio then no longer matches, so remove it (not needed for text models).
    t_inst = time.time()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], capture_output=True, text=True)
    ok, msg = vllm_import_check(sys.executable)
    BACKEND["route1"] = dict(install_ok=r.returncode == 0, import_ok=ok, minutes=round((time.time() - t_inst) / 60, 1), msg=msg[-300:])
    print("route 1 (system python):", "OK" if ok else "FAILED", "\n", msg)
    if ok:
        pythons.append(sys.executable)
    else:
        # Route 2: an isolated virtual environment, so nothing preinstalled can clash with vLLM's own torch
        t_inst = time.time()
        venv = "/tmp/vllm_env"
        subprocess.run([sys.executable, "-m", "venv", venv], check=False)
        py = os.path.join(venv, "bin", "python")
        subprocess.run([py, "-m", "pip", "install", "-q", "--upgrade", "pip"], capture_output=True)
        r2 = subprocess.run([py, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
        ok2, msg2 = vllm_import_check(py)
        BACKEND["route2"] = dict(install_ok=r2.returncode == 0, import_ok=ok2, minutes=round((time.time() - t_inst) / 60, 1), msg=msg2[-300:])
        print("route 2 (isolated venv):", "OK" if ok2 else "FAILED", "\n", msg2, "\n", r2.stderr[-800:])
        if ok2:
            pythons.append(py)
    for py in pythons:
        for extra_env, extra_args in (({}, []), ({}, ["--enforce-eager"]), ({"VLLM_ATTENTION_BACKEND": "TRITON_ATTN"}, ["--enforce-eager"])):
            ok, p = launch_vllm(py, extra_env, extra_args)
            BACKEND["attempts"].append(dict(python=py, env=extra_env, args=extra_args, ok=ok, log_tail=None if ok else tail("vllm.log", 15)[-1500:]))
            if ok:
                BACKEND.update(kind="vllm", proc=p, model=CFG["model"]); break
            print("vLLM launch failed:", extra_env, extra_args, "\n", tail("vllm.log", 15))
        if BACKEND["kind"]:
            break
if BACKEND["kind"] is None:
    if CFG["backend"] == "vllm":
        raise RuntimeError("vLLM requested but could not start; see BACKEND['attempts']")
    p = subprocess.Popen([sys.executable, "simserver.py", "--port", "8000"], stdout=open("sim.log", "w"), stderr=subprocess.STDOUT)
    assert wait_http("http://127.0.0.1:8000/health", 60, p), tail("sim.log")
    BACKEND.update(kind="sim", proc=p, model="sim")
print("backend:", BACKEND["kind"], [(a["env"], a["args"], a["ok"]) for a in BACKEND["attempts"]])


In [ ]:
# Cell 6 — gateway + smoke test through the real OpenAI SDK
for f in ("gateway_log.csv",):
    if os.path.exists(f): os.remove(f)
gw = subprocess.Popen([sys.executable, "gateway.py", "--port", "9000", "--backend", "http://127.0.0.1:8000", "--log", "gateway_log.csv"],
                      stdout=open("gateway.out", "w"), stderr=subprocess.STDOUT)
assert wait_http(CFG["gateway"] + "/v1/models", 60, gw), tail("gateway.out")
from openai import OpenAI
c = OpenAI(api_key="smoke-000", base_url=CFG["gateway"] + "/v1")
t = time.time()
s = c.chat.completions.create(model=BACKEND["model"], messages=[{"role": "user", "content": "Say hello."}], max_tokens=96, stream=True,
                              extra_body={"ignore_eos": True})
n = sum(1 for _ in s)
BACKEND["smoke"] = dict(chunks=n, seconds=round(time.time() - t, 2))
print("smoke test:", BACKEND["smoke"])


In [ ]:
# Cell 7 — run every population at once, each in its own process
for f in glob.glob("clientlog_*.csv") + glob.glob("labels_*.json") + glob.glob("driverstats_*.json"):
    os.remove(f)
CFG["t_start"] = time.time() + 30                    # common start, after every process has imported its libraries
dcfg = dict(CFG, model=BACKEND["model"])
json.dump(dcfg, open("driver_cfg.json", "w"))
procs = {p: subprocess.Popen([sys.executable, "driver6.py", "driver_cfg.json", p], stdout=open(f"drv_{p}.out", "w"), stderr=subprocess.STDOUT)
         for p in CFG["pops"]}
t_run = time.time()
while any(pr.poll() is None for pr in procs.values()):
    time.sleep(120)
    nlines = sum(1 for _ in open("gateway_log.csv")) - 1
    print(f"[run] t={time.time() - CFG['t_start']:.0f}s  gateway requests={nlines}  running={sum(pr.poll() is None for pr in procs.values())}", flush=True)
RUN = dict(rc={p: pr.returncode for p, pr in procs.items()}, wall_min=round((time.time() - t_run) / 60, 1),
           driver_tail={p: open(f"drv_{p}.out").read().strip().splitlines()[-1][:300] if open(f"drv_{p}.out").read().strip() else "" for p in procs})
for p, line in RUN["driver_tail"].items(): print(line)


In [ ]:
# Cell 8 — analysis (labels are used only to say which keys form a pool and to score the answers)
import numpy as np, pandas as pd
import importlib, occlib, fplib
importlib.reload(occlib); importlib.reload(fplib)
from occlib import occupancy, fano_khat
rng = np.random.default_rng(CFG["seed"])
t_an = time.time()
labels = {}
for f in glob.glob("labels_*.json"): labels.update(json.load(open(f)))
log = pd.read_csv("gateway_log.csv", dtype={"rid": str})
log["grp"] = log.key.map(lambda k: labels.get(k, {}).get("pop"))
log.loc[log.grp.isin(["hum_a", "hum_b"]), "grp"] = "hum"
log["pos"] = log.key.map(lambda k: labels.get(k, {}).get("pos"))
okm = (log.status == 200) & log.grp.notna()
RES = {"cfg": {k: v for k, v in CFG.items() if k not in ("expected",)}, "backend": {k: v for k, v in BACKEND.items() if k != "proc"},
       "run": RUN, "log": dict(n=len(log), ok_frac=round(float(okm.mean()), 4), by_pop=log[okm].grp.value_counts().to_dict())}
t0 = CFG["t_start"] + CFG["warmup_s"]; t1 = min(float(log.t_end.max()), CFG["t_start"] + CFG["run_s"]) - 30
d = log[okm & (log.t_end > t0) & (log.t_arrival < t1)].copy()
d["dur"] = d.t_end - d.t_arrival
RES["window_s"] = round(t1 - t0)
RES["durations_s"] = d.groupby("grp").dur.median().round(2).to_dict()
keys_by = {g: sorted(v) for g, v in d.groupby("grp").key.unique().items()}

def pool_arrays(keys_):
    keys_ = sorted(keys_); idx = {k: i for i, k in enumerate(keys_)}
    m = d.key.isin(idx)
    return d.key[m].map(idx).values.astype(np.int64), d.t_arrival.values[m], d.t_end.values[m], keys_

# ---- 8a. client <-> gateway timing (confound diagnostics)
cl = []
for f in glob.glob("clientlog_*.csv"):
    x = pd.read_csv(f, dtype={"rid": str}); x["src"] = f[len("clientlog_"):-4]; cl.append(x)
cl = pd.concat(cl, ignore_index=True)
j = d.merge(cl, on="rid", how="inner")
q = lambda v: {f"p{int(p * 100)}": round(float(np.quantile(v, p)) * 1000, 1) for p in (0.5, 0.9, 0.99)} if len(v) else {}
RES["timing_ms"] = dict(joined_frac=round(len(j) / max(len(d), 1), 3),
                        client_send_to_gateway_arrival=q((j.t_arrival - j.t_send).values),
                        gateway_end_to_client_done=q((j.t_done - j.t_end).values),
                        relay_schedule_lag={g: q((j.t_send - j.t_sched)[j.grp == g].values) for g in ("P7_relay_random", "P8_relay_rr")})

# ---- 8b. human null for R-type statistics
hum = keys_by.get("hum", []); bat = keys_by.get("bat", [])
RKEYS = ["R", "R5", "R20", "R120"]
def r_stats(a, s, e, M):
    out = {"R": float(occupancy(a, s, e, t0, t1, None, n_accounts=M)["R"])}
    for tau in (5.0, 20.0, 120.0):
        out[f"R{int(tau)}"] = float(occupancy(a, s, e, t0, t1, tau, n_accounts=M)["R"])
    return out
null = []
for _ in range(200):
    a, s, e, ks = pool_arrays(list(rng.choice(hum, min(40, len(hum)), replace=False)))
    null.append(r_stats(a, s, e, len(ks)))
NULL = {k: np.array([x[k] for x in null]) for k in RKEYS}
RES["null_humans_40"] = {k: dict(mean=round(float(v.mean()), 4), sd=round(float(v.std()), 4)) for k, v in NULL.items()}
def zR(st):   # positive = turn-taking (below the human null), negative = bunching
    return {k: float((NULL[k].mean() - st[k]) / NULL[k].std()) for k in RKEYS if k in st}
# human pools through the full fingerprint (no surrogates): reference values for ring / conc / coll
href = []
for _ in range(20):
    a, s, e, ks = pool_arrays(list(rng.choice(hum, min(40, len(hum)), replace=False)))
    f, _ = fplib.fingerprint(a, s, e, len(ks), t0, t1, rng, n_sur=0)
    href.append(f)
RES["human_reference"] = {k: dict(mean=round(float(np.nanmean([h[k] for h in href])), 4), q95=round(float(np.nanquantile([h[k] for h in href], 0.95)), 4))
                          for k in ("ring", "conc", "coll")}

# ---- 8c. fingerprint every pool, classify blind, score
TRUE_K = {p: CFG["pops"][p].get("K") for p in CFG["pops"] if CFG["pops"][p]["kind"] not in ("human", "batch", "open_loop")}
pools = {p: keys_by[p] for p in CFG["pops"] if p in keys_by and CFG["pops"][p]["kind"] not in ("human", "batch")}
pools["hum_random40"] = list(rng.choice(hum, min(40, len(hum)), replace=False))
pools["benign_36hum_4bat"] = list(rng.choice(hum, min(36, len(hum)), replace=False)) + list(bat)
rows = {}
for name, ks_ in pools.items():
    a, s, e, ks = pool_arrays(ks_); M = len(ks)
    f, order = fplib.fingerprint(a, s, e, M, t0, t1, rng, n_sur=40, occupancy_fn=occupancy)
    z = zR(f); cls = fplib.classify(f, z)
    fk = fano_khat(a, s, e, t0, t1)
    used = np.unique(a); busy = [float(np.sum(np.minimum(e[a == k], t1) - np.maximum(s[a == k], t0)) / (t1 - t0)) for k in used]
    true_pos = np.array([labels[k]["pos"] for k in ks]) if all(k in labels for k in ks) else np.arange(M)
    rows[name] = dict(expected=CFG["expected"].get(name), predicted=cls, correct=cls == CFG["expected"].get(name),
                      n_req=f["n_req"], R=round(f["R"], 3), R5=round(f["R5"], 3), R20=round(f["R20"], 3), R120=round(f["R120"], 3),
                      z_R=round(z["R"], 1), z_R20=round(z["R20"], 1), coll=round(f["coll"], 2), ring_z=round(f["ring_z"], 1),
                      conc_z=round(f["conc_z"], 1) if np.isfinite(f["conc_z"]) else None, lag_win=f["lag_win"],
                      ring_neighbour_acc=round(fplib.neighbour_accuracy(order, np.argsort(np.argsort(true_pos))), 2),
                      K_true=TRUE_K.get(name), K_fano=round(float(fk["K_hat"]), 1) if np.isfinite(fk["K_hat"]) else None,
                      K_little=round(f["K_little"], 1), think_s=round(f["think_s"], 2),
                      keys_used=int(len(used)), mean_inflight_per_used_key=round(float(np.mean(busy)), 2))
    print(f"{name:18s} -> {cls:38s} {'OK' if rows[name]['correct'] else 'WRONG'}", flush=True)
RES["pools"] = rows

# ---- 8d. ring robustness: shorter windows and partial observation (only half of the ring's keys at this provider)
rob = {}
for name in ("P3_rrshared", "P5_rrjitter", "P8_relay_rr"):
    if name not in keys_by: continue
    out = {}
    for w in (600, 1200):
        a, s, e, ks = pool_arrays(keys_by[name]); m = (s < t0 + w)
        f, order = fplib.fingerprint(a[m], s[m], e[m], len(ks), t0, min(t1, t0 + w), rng, n_sur=20)
        out[f"first_{w}s"] = dict(ring_z=round(f["ring_z"], 1), acc=round(fplib.neighbour_accuracy(order, np.arange(len(ks))), 2))
    accs, zs = [], []
    for _ in range(5):
        sub = sorted(rng.choice(keys_by[name], len(keys_by[name]) // 2, replace=False))
        a, s, e, ks = pool_arrays(sub)
        f, order = fplib.fingerprint(a, s, e, len(ks), t0, t1, rng, n_sur=20)
        tp = np.array([labels[k]["pos"] for k in ks])
        accs.append(fplib.neighbour_accuracy(order, np.argsort(np.argsort(tp)))); zs.append(f["ring_z"])
    out["half_of_keys"] = dict(ring_z_median=round(float(np.median(zs)), 1), acc_median=round(float(np.median(accs)), 2))
    rob[name] = out
RES["ring_robustness"] = rob

# ---- 8e. site-wide blind discovery: successor graph over every key, no labels, BH over all key pairs
allk = sorted(d.key.unique()); idx = {k: i for i, k in enumerate(allk)}
A = d.key.map(idx).values.astype(np.int64)
comp, edges = fplib.site_handoff_graph(A, d.t_arrival.values, d.t_end.values, len(allk), t0, t1)
grp_of = np.array([labels[k]["pop"] if labels[k]["pop"] not in ("hum_a", "hum_b") else "hum" for k in allk])
comps = []
for c in np.unique(comp[comp >= 0]):
    mem = grp_of[comp == c]
    if len(mem) >= 3:
        u, n = np.unique(mem, return_counts=True); comps.append(dict(size=int(len(mem)), members=dict(zip(u.tolist(), n.tolist()))))
comps.sort(key=lambda c: -c["size"])
recovery = {}
for p in pools:
    if p.startswith(("hum", "benign")): continue
    tot = (grp_of == p).sum()
    best = max([c["members"].get(p, 0) for c in comps] or [0])
    host = next((c for c in comps if c["members"].get(p, 0) == best and best > 0), None)
    recovery[p] = dict(recall=round(best / tot, 2) if tot else None, purity=round(best / host["size"], 2) if host else None)
RES["site_graph"] = dict(n_keys=len(allk), n_edges=len(edges), humans_in_components=int(((comp >= 0) & (grp_of == "hum")).sum()),
                         components=comps[:12], recovery=recovery)

# ---- 8f. pre-registered scorecard
pr = RES["pools"]
closed = [p for p in ("P1_shuffle", "P2_least", "P3_rrshared", "P4_rrworker", "C_company") if p in pr]
RES["scorecard"] = dict(
    S1_classification=f"{sum(r['correct'] for r in pr.values())}/{len(pr)} correct (pass if >= {len(pr) - 2})",
    S2_ring_recovery={p: pr[p]["ring_neighbour_acc"] for p in ("P3_rrshared", "P5_rrjitter", "P8_relay_rr") if p in pr},
    S3_K_little_within_25pct={p: bool(pr[p]["K_little"] and abs(pr[p]["K_little"] / pr[p]["K_true"] - 1) <= 0.25) for p in closed},
    S3b_K_fano_within_25pct={p: bool(pr[p]["K_fano"] and abs(pr[p]["K_fano"] / pr[p]["K_true"] - 1) <= 0.25) for p in closed},
    S4_site_graph={p: RES["site_graph"]["recovery"].get(p) for p in ("P3_rrshared", "P4_rrworker", "P5_rrjitter", "P8_relay_rr")},
    S4_humans_in_components=RES["site_graph"]["humans_in_components"],
    S5_relay_random_zR=pr.get("P7_relay_random", {}).get("z_R"), S5_heavy_benign_zR20=pr.get("benign_36hum_4bat", {}).get("z_R20"))
RES["analysis_min"] = round((time.time() - t_an) / 60, 1)
print(json.dumps(RES["scorecard"], indent=1))


In [ ]:
# Cell 9 — shut down and print the summary
for proc in (gw, BACKEND.get("proc")):
    try: proc.terminate()
    except Exception: pass
d.to_csv("nb6_requests.csv", index=False)
RES["runtime_min"] = round((time.time() - T_START) / 60, 1)
json.dump(RES, open("nb6_results.json", "w"), indent=1, default=str)
print("===== PASTE FROM HERE =====")
print(json.dumps(RES, separators=(",", ":"), default=str))
print("===== PASTE TO HERE =====")
